# Olist delivery KPIs and seller defect rate

This notebook walks through the whole analysis with the outputs shown under each step.
The same logic runs as plain scripts in `src/` (`python run_all.py` builds the Excel report).

Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). Put the CSV files in `data/raw/` before running.

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("src")
from load_data import load_tables

pd.set_option("display.float_format", "{:,.2f}".format)
tables = load_tables()
orders, items, reviews = tables["orders"], tables["items"], tables["reviews"]
customers, sellers = tables["customers"], tables["sellers"]

## 1. First look

Size of each table and whether the key columns are unique. If a key repeats, a join on it will duplicate rows.

In [ ]:
keys = {"orders": "order_id", "items": "order_id", "reviews": "order_id",
        "customers": "customer_id", "sellers": "seller_id"}
pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1],
     "key": keys[name], "duplicate keys": df[keys[name]].duplicated().sum()}
    for name, df in tables.items()
])

`items` repeats `order_id` on purpose (one order can have several products). `reviews` should not: some orders have more than one review, so they need to be cleaned before the join.

## 2. Data quality checks

In [ ]:
delivered = orders["order_status"] == "delivered"
no_date = orders["order_delivered_customer_date"].isna()
carrier_late = orders["order_delivered_carrier_date"] > orders["order_delivered_customer_date"]

checks = pd.DataFrame([
    ("delivered but no delivery date", (delivered & no_date).sum()),
    ("delivered before it was bought",
     (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum()),
    ("given to carrier after it reached the customer", carrier_late.sum()),
    ("price zero or below", (items["price"] <= 0).sum()),
    ("review score outside 1-5", (~reviews["review_score"].between(1, 5)).sum()),
    ("orders with more than one review", (reviews.groupby("order_id").size() > 1).sum()),
    ("orders with more than one seller", (items.groupby("order_id")["seller_id"].nunique() > 1).sum()),
], columns=["check", "rows"])
checks

Decisions: keep only delivered orders with a delivery date and a logical order of dates, keep the latest review per order, and leave multi-seller orders out of the seller score (one delivery date cannot be split between sellers).

## 3. Join the tables and check the row count after every join

In [ ]:
base = orders[delivered & ~no_date & ~carrier_late].copy()
reviews_one = (reviews.sort_values("review_answer_timestamp")
                      .drop_duplicates(subset="order_id", keep="last"))

rows = len(base)
base = base.merge(customers[["customer_id", "customer_state"]],
                  on="customer_id", how="left", validate="one_to_one")
print(f"orders + customers: {rows:,} -> {len(base):,}")

rows = len(base)
base = base.merge(reviews_one[["order_id", "review_score"]],
                  on="order_id", how="left", validate="one_to_one")
print(f"orders + reviews:   {rows:,} -> {len(base):,}")

delivered_day = base["order_delivered_customer_date"].dt.normalize()
base["delivery_days"] = (base["order_delivered_customer_date"] - base["order_purchase_timestamp"]).dt.days
base["delay_days"] = (delivered_day - base["order_estimated_delivery_date"]).dt.days
base["is_late"] = base["delay_days"] > 0
base["month"] = base["order_purchase_timestamp"].dt.to_period("M").astype(str)
print(f"orders in the analysis: {len(base):,}")

## 4. Delivery KPIs

In [ ]:
late = base[base["is_late"]]
pd.Series({
    "On-time rate (%)": 100 * (1 - base["is_late"].mean()),
    "Average delivery time (days)": base["delivery_days"].mean(),
    "Median delivery time (days)": base["delivery_days"].median(),
    "Late orders: average days late": late["delay_days"].mean(),
    "On-time orders: average days early": -base.loc[~base["is_late"], "delay_days"].mean(),
}, name="value").round(1).to_frame()

In [ ]:
monthly = base.groupby("month").agg(orders=("order_id", "count"), late_pct=("is_late", "mean"))
monthly = monthly[monthly["orders"] >= 500]
average = 100 * base["is_late"].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, 100 * monthly["late_pct"], marker="o", color="#1F3864")
ax.axhline(average, color="grey", linestyle="--", label=f"Average ({average:.1f}%)")
ax.set_title("Late deliveries by month of purchase")
ax.set_ylabel("Late orders (%)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.show()

Peaks: November 2017 (Black Friday volume) and February to March 2018. The data does not say why the 2018 peak happened, so it is a question for the logistics team rather than a conclusion.

In [ ]:
by_state = base.groupby("customer_state").agg(
    orders=("order_id", "count"),
    late_pct=("is_late", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
)
by_state = by_state[by_state["orders"] >= 500]
by_state["late_pct"] *= 100
by_state.sort_values("late_pct", ascending=False).head(10).round(1)

## 5. Do late deliveries hurt reviews?

In [ ]:
rated = base[base["review_score"].notna()].copy()
bins = [float("-inf"), -7, -1, 0, 3, 7, 14, float("inf")]
labels = ["7+ days early", "1-6 days early", "on the day", "1-3 days late",
          "4-7 days late", "8-14 days late", "15+ days late"]
rated["delay_group"] = pd.cut(rated["delay_days"], bins=bins, labels=labels)

by_delay = rated.groupby("delay_group", observed=True).agg(
    orders=("order_id", "count"),
    avg_review=("review_score", "mean"),
    bad_review_pct=("review_score", lambda s: 100 * (s <= 2).mean()),
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_delay.index.astype(str), by_delay["avg_review"], color="#0B6E69")
for x, y in zip(by_delay.index.astype(str), by_delay["avg_review"]):
    ax.text(x, y + 0.08, f"{y:.2f}", ha="center")
ax.set_ylim(0, 5)
ax.set_title("Average review score by delivery delay")
ax.set_ylabel("Average review (1-5)")
ax.tick_params(axis="x", rotation=30)
plt.show()
by_delay.round(2)

In [ ]:
bad = rated["review_score"] <= 2
print(f"late orders are {rated['is_late'].mean():.1%} of rated orders")
print(f"but {rated.loc[bad, 'is_late'].mean():.1%} of all 1-2 star reviews")

## 6. Seller defect rate (DPMO) and audit list

Each single-seller order has two chances to go wrong: a late delivery and a bad review (1-2 stars, if the customer left one).
DPMO = defects / opportunities x 1,000,000. The network DPMO is total defects divided by total opportunities, not an average of seller rates.
Sellers with 30+ orders and a DPMO above 1.5x the network are flagged, then ranked by excess defects (defects above what the network rate would give at their volume).

In [ ]:
multi = items.groupby("order_id")["seller_id"].nunique() > 1
single = (items[~items["order_id"].isin(multi[multi].index)]
          .drop_duplicates(["order_id", "seller_id"])
          .merge(base[["order_id", "is_late", "review_score"]], on="order_id", how="inner"))

seller = single.groupby("seller_id").agg(
    orders=("order_id", "count"),
    late=("is_late", "sum"),
    rated=("review_score", lambda s: s.notna().sum()),
    bad_reviews=("review_score", lambda s: (s <= 2).sum()),
)
seller["opportunities"] = seller["orders"] + seller["rated"]
seller["defects"] = seller["late"] + seller["bad_reviews"]
seller["dpmo"] = seller["defects"] / seller["opportunities"] * 1_000_000

network = seller["defects"].sum() / seller["opportunities"].sum() * 1_000_000
threshold = 1.5 * network
active = seller[seller["orders"] >= 30].copy()
active["flag"] = active["dpmo"] > threshold
active["excess_defects"] = active["defects"] - active["opportunities"] * network / 1_000_000

flagged = active[active["flag"]]
print(f"network DPMO {network:,.0f}, audit threshold {threshold:,.0f}")
print(f"flagged {len(flagged)} of {len(active)} sellers, "
      f"{flagged['defects'].sum() / active['defects'].sum():.1%} of their defects")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ok = active[~active["flag"]]
ax.scatter(ok["orders"], ok["dpmo"], s=15, alpha=0.5, color="grey", label="Below threshold")
ax.scatter(flagged["orders"], flagged["dpmo"], s=25, color="#C00000", label="Flagged for audit")
ax.axhline(threshold, color="#C00000", linestyle="--", label=f"Audit threshold ({threshold:,.0f})")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:,.0f}"))
ax.set_title("Seller DPMO vs number of orders")
ax.set_xlabel("Orders (log scale)")
ax.set_ylabel("DPMO")
ax.legend()
plt.show()

flagged.sort_values("excess_defects", ascending=False)[
    ["orders", "late", "bad_reviews", "dpmo", "excess_defects"]].head(10).round(1)

## Summary

- About 93% of orders arrive on time, but late orders are late by more than 10 days on average.
- The review score drops step by step with the delay. Late orders are under 7% of rated orders but about a third of all 1-2 star reviews.
- Flagged sellers cause around a fifth of seller defects. Audits help, but most defects are spread across the network, so better delivery estimates and carrier work in the worst states matter more.
- Small sellers vary a lot by chance (the funnel shape above), so a flag should be confirmed in a second period before an audit.